In [239]:
import pandas as pd
import numpy as np
import matplotlib as mtp

In [240]:
df_gdp = pd.read_csv('arquivos_base/gdp.csv', decimal='.', thousands=',')

In [241]:
df_gdp

,Country,Region,Year,GDP_pp
0,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1901,613.99
1,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1906,624.04
2,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1911,634.25
3,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1916,647.28
4,Afghanistan,"Middle East, North Africa, and Greater Arabia",1/1/1921,662.40
...,...,...,...,...
4414,Zimbabwe,Sub-Saharan Africa,1/1/1991,782.09
4415,Zimbabwe,Sub-Saharan Africa,1/1/1996,781.50
4416,Zimbabwe,Sub-Saharan Africa,1/1/2001,719.96
4417,Zimbabwe,Sub-Saharan Africa,1/1/2006,520.17


In [242]:
df_gdp.info()

<class 'pandas.DataFrame'>
RangeIndex: 4419 entries, 0 to 4418
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Country   4419 non-null   str    
 1   Region    4419 non-null   str    
 2   Year      4419 non-null   str    
 3    GDP_pp   4419 non-null   float64
dtypes: float64(1), str(3)
memory usage: 138.2 KB


In [243]:
df_gdp.columns = 'Country', 'Region', 'Year', 'GDP_pp'
# erro ao chamar coluna 'GDP_pp'. Resolvido declarando colunas

In [244]:
df_gdp['Year'] = df_gdp['Year'].apply(lambda x: int(x.split('/')[2]))
# transformando coluna Year para INT, para facilitar operacao

# INFORME O PRIMEIRO VALOR REGISTRADO DE CADA PAIS

In [245]:
df_gdp['Year'].value_counts()
# nao ha uma confirmacao de aparicoes exatas dos paises por ano, oscila a quantidade de aparicoes conforme o ano.
# portanto na primeira amostragem ha risco de ter algum pais que inicie sua amostragem em um ano diferente dos demais.

Year
1991    193
1996    193
2001    193
1901    192
1906    192
1911    192
1916    192
1921    192
1926    192
1931    192
1936    192
1941    192
1946    192
1951    192
1956    192
1961    192
1966    192
1971    192
1976    192
1981    192
1986    192
2006    192
2011    192
Name: count, dtype: int64

In [246]:
df_gdp.groupby('Country').agg({'Year':'min'}).reset_index().sort_values(by='Year')
# um dos paises inicia a amostragem em 1991, diferente dos outros paises. Pais -> 'Kosovo'

,Country,Year
0,Afghanistan,1901
1,Albania,1901
2,Algeria,1901
3,Andorra,1901
4,Angola,1901
...,...,...
188,Venezuela,1901
189,Vietnam,1901
190,"Yemen, Rep.",1901
191,Zambia,1901


In [247]:
df_ano_minimo = df_gdp.groupby('Country').agg({'Year':'min', 'GDP_pp':'first'}).reset_index()

In [248]:
df_ano_minimo
# amostragem do primeiro valor registrado de cada pais

,Country,Year,GDP_pp
0,Afghanistan,1901,613.99
1,Albania,1901,1062.01
2,Algeria,1901,1807.76
3,Andorra,1901,3352.50
4,Angola,1901,525.76
...,...,...,...
188,Venezuela,1901,766.21
189,Vietnam,1901,572.96
190,"Yemen, Rep.",1901,729.39
191,Zambia,1901,532.38


# Informe as regioes com maior crescimento de PIB per capita no seculo passado

In [249]:
df_gdp['Year'].value_counts().sort_index()
# seculo passado ==  1996 <=

Year
1901    192
1906    192
1911    192
1916    192
1921    192
1926    192
1931    192
1936    192
1941    192
1946    192
1951    192
1956    192
1961    192
1966    192
1971    192
1976    192
1981    192
1986    192
1991    193
1996    193
2001    193
2006    192
2011    192
Name: count, dtype: int64

In [250]:
df_gdp_crescimento = df_gdp.groupby(['Year', 'Region']).agg({'GDP_pp':'mean'}).reset_index()

In [251]:
df_gdp_crescimento = df_gdp_crescimento.loc[(df_gdp_crescimento['Year'] == 1901) | (df_gdp_crescimento['Year'] == 1996)]
# selecionando dados de inicio e final (ate encerramento do seculo) para retornar a diferenca de valores

In [252]:
df_gdp_crescimento.rename(columns={'GDP_pp':'GDP_inicial'}, inplace=True)
# formatacao do DF

In [253]:
df_operacional = df_gdp_crescimento[df_gdp_crescimento['Year'] == 1996].reset_index()
# criando um DF de manipulacao dos dados

In [254]:
df_gdp_crescimento.drop(df_gdp_crescimento[df_gdp_crescimento['Year'] == 1996].index, axis=0, inplace=True)
df_gdp_crescimento.drop(columns='Year', inplace=True)
# formatacao

In [255]:
df_gdp_crescimento['GDP_final'] = df_operacional['GDP_inicial']
# alinhando valores pra executar diferenca de valores

In [256]:
df_gdp_crescimento['GDP_diferenca'] = df_gdp_crescimento['GDP_final'] - df_gdp_crescimento['GDP_inicial']
# formulando a operacao de diferenca

In [257]:
df_gdp_crescimento.sort_values(by='GDP_diferenca').reset_index().drop(columns='index').tail(3)
# As 3 regioes com os maiores crescimentos de PIB no seculo passado (ordem crescente)

,Region,GDP_inicial,GDP_final,GDP_diferenca
5,"Middle East, North Africa, and Greater Arabia",1164.350000,11145.343913,9980.993913
6,Europe,2583.788478,17932.684894,15348.896415
7,North America,3676.015000,25355.687500,21679.672500


# Preencha os anos ausentes em cada pais com uma estimativa, baseada na diferenca entre o proximo registro e o anterior

In [ ]:
df_gdp.sort_values(by=['Year', 'Country'], inplace=True)

,Country,Region,Year,GDP_pp
0,Afghanistan,"Middle East, North Africa, and Greater Arabia",1901,613.99
23,Albania,Europe,1901,1062.01
46,Algeria,"Middle East, North Africa, and Greater Arabia",1901,1807.76
69,Andorra,Europe,1901,3352.50
92,Angola,Sub-Saharan Africa,1901,525.76
...,...,...,...,...
4326,Venezuela,South America,2011,11241.08
4349,Vietnam,Asia,2011,2961.83
4372,"Yemen, Rep.","Middle East, North Africa, and Greater Arabia",2011,2102.22
4395,Zambia,Sub-Saharan Africa,2011,1480.69
